In [1]:
import pandas as pd
import json
import random
import re
import numpy as np

In [2]:
UNCERTAINITY_RESPONSES_PATH = "/chronos_data/avirinchipur/reasoning_for_psych/expts/responses/expt_gpt-5.uncertainity_deptext_4codes_3subjects_v1.gen_v3.csv"
responses_df = pd.read_csv(UNCERTAINITY_RESPONSES_PATH)

In [3]:
responses_df

,user_id,input_text,target_value,response_packet,user_text
0,1,"[{'role': 'system', 'content': 'You are a help...",0.0,"{""id"": ""resp_68cf6a05398481a2ac3efd6295e90a230...","Over the past two weeks, I have not been depre..."
1,2,"[{'role': 'system', 'content': 'You are a help...",17.0,"{""id"": ""resp_68cf6a1e9c808190a209bd8c8e2910c40...",I don't think I have felt depressed. I'm not f...
2,3,"[{'role': 'system', 'content': 'You are a help...",21.0,"{""id"": ""resp_68cf6a376fe081949137bee5b7ebd3da0...",I have been depressed because I am worried abo...
3,4,"[{'role': 'system', 'content': 'You are a help...",13.0,"{""id"": ""resp_68cf6a521d0481a0877e5760a86a77880...",Over the past 2 weeks i have been feeling depr...
4,5,"[{'role': 'system', 'content': 'You are a help...",9.0,"{""id"": ""resp_68cf6a5e3bac81a18f4a70d5926a2cc90...",yes very much just seem everything is getting ...
...,...,...,...,...,...
951,972,"[{'role': 'system', 'content': 'You are a help...",11.0,"{""id"": ""resp_68cfe45c867081a3aab1a82e5e8a03ee0...",There has been time when I have not wanted to ...
952,973,"[{'role': 'system', 'content': 'You are a help...",20.0,"{""id"": ""resp_68d1bc73afe8819ebcf19df0994c83c70...","I’m not sure if I’ve been depressed exactly, I..."
953,974,"[{'role': 'system', 'content': 'You are a help...",20.0,"{""id"": ""resp_68cfe5234a208192ade1d4e5280e5a490...","I haven’t felt depressed. However, I have expe..."
954,975,"[{'role': 'system', 'content': 'You are a help...",22.0,"{""id"": ""resp_68cfe53d07a8819493dad092621984dc0...",I have suffered from depression since my early...


In [4]:
# json.loads(json.loads(responses_df.iloc[1]['response_packet'])['output'][1]['content'][0]['text'])
json.loads(responses_df.iloc[1]['response_packet'])

{'id': 'resp_68cf6a1e9c808190a209bd8c8e2910c40bbc93a583ae2155',
 'created_at': 1758423582.0,
 'error': None,
 'incomplete_details': None,
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5-2025-08-07',
 'object': 'response',
 'output': [{'id': 'rs_68cf6a1f1e10819098c8dbdca0c9348c0bbc93a583ae2155',
   'summary': [{'text': 'I’m thinking about how the user wants multiple code-evidence pairs. I should provide both hedged expressions and neutral statements like “I’m not familiar…” to show some level of certainty. While that’s not directly saying “not depressed,” it does serve as evidence. I realize they may want only one overall code, but they mentioned that multiple pairs are acceptable. So, I’ll include two code-evidence pairs and make sure to keep everything clear.',
     'type': 'summary_text'},
    {'text': "I’m considering the user's request, noting they want a single code that best captures the overall stance. Even though they said multiple codes are possible, I think I should 

In [5]:
# --- Robust JSON extractor ---
def extract_annotation_packet(text):
    """
    Locate and extract a JSON object that contains the key "Depression" from noisy text.
    Returns a Python dict on success, raises ValueError on failure.
    """
    # 1) find anchor
    anchor = '"Depression"'
    pos = text.find(anchor)
    if pos == -1:
        alt = re.search(r"['\"]?Depression['\"]?\s*:", text)
        if not alt:
            raise ValueError("No 'Depression' anchor found in text.")
        pos = alt.start()

    # 2) find the nearest opening brace before the anchor
    start = text.rfind('{', 0, pos)
    if start == -1:
        start = text.find('{', pos)
        if start == -1:
            raise ValueError("Could not locate an opening brace for JSON object.")

    # 3) walk forward and balance braces
    depth = 0
    end = -1
    for i in range(start, len(text)):
        ch = text[i]
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end == -1:
        last = text.rfind('}')
        if last != -1 and last > start:
            candidate = text[start:last+1]
        else:
            raise ValueError("Couldn't find a matching closing brace for JSON object.")
    else:
        candidate = text[start:end]

    # 4) try parsing
    try:
        return json.loads(candidate)
    except Exception:
        # heuristic: replace single quotes with double quotes
        candidate_fixed = candidate.replace("'", '"')
        return json.loads(candidate_fixed)
    
# Define helper to transform packet into required row
def parse_packet_to_row(user_id, packet):
    row = {"user_id": user_id}
    # Initialize all boolean flags as 0
    for subject in ["Depression", "Symptom", "Comorbid"]:
        for code_name in ["weakener", "neutral", "wkndstrgthnr", "strength"]:
            row[f"is_{code_name}_{subject.lower()}"] = 0
        row[f"evidences_{subject.lower()}"] = []
    
    # Map codes to names
    code_map = {
        0: "weakener",
        1: "neutral",
        2: "wkndstrgthnr",
        3: "strength"
    }
    
    # Populate values
    for subject in ["Depression", "Symptom", "Comorbid"]:
        for entry in packet.get(subject, []):
            code = entry["code"]
            evidence = entry["evidence"]
            if code in code_map:
                row[f"is_{code_map[code]}_{subject.lower()}"] = 1
            row[f"evidences_{subject.lower()}"].append(evidence)
    
    # Join evidences as a list-like string for CSV
    for subject in ["Depression", "Symptom", "Comorbid"]:
        row[f"evidences_{subject.lower()}"] = json.dumps(row[f"evidences_{subject.lower()}"])
    
    return row

In [6]:
skipped_rows = []
parsed_responses_df = []
for idx, row in responses_df.iterrows():
    user_id = row['user_id']
    response_packet = json.loads(row['response_packet'])
    if response_packet['incomplete_details'] is not None:
        skipped_rows.append(user_id)
        continue
    text_packet = response_packet['output'][1]['content'][0]['text']
    packet = extract_annotation_packet(text_packet)
    row = parse_packet_to_row(user_id=user_id, packet=packet)

    # Convert to DataFrame
    parsed_row_df = pd.DataFrame([row])
    parsed_responses_df.append(parsed_row_df)
parsed_responses_df = pd.concat(parsed_responses_df, ignore_index=True, axis=0)

In [7]:
responses_df[responses_df.user_id.isin(skipped_rows)]['response_packet'].apply(lambda x: json.loads(x)['incomplete_details']['reason']).value_counts()

max_output_tokens    10
Name: response_packet, dtype: int64

In [8]:
responses_df[responses_df.user_id.isin(skipped_rows)]['response_packet'].apply(lambda x: json.loads(x)).iloc[0]

{'id': 'resp_68d1ccf0dc148191b18166ffd84198a004efb82dd57d567f',
 'created_at': 1758579952.0,
 'error': None,
 'incomplete_details': {'reason': 'max_output_tokens'},
 'instructions': None,
 'metadata': {},
 'model': 'gpt-5-2025-08-07',
 'object': 'response',
 'output': [{'id': 'rs_68d1ccf1836481918d1bfb6aa14d679a04efb82dd57d567f',
   'summary': [{'text': '**Evaluating depression coding**\n\nIt says "For Depression, assign the single code that best reflects..." so I know we should include one element. The format shows an array, but I can focus on the strongest phrase. I’ll include just one for Depression with code 3 and the initial sentence. \n\nSymptoms noted: the PHQ-9 mentions loss of interest or pleasure. The statement about not looking forward to the future suggests anhedonia, which relates to hopelessness but fits under loss of interest. I\'ll decide whether to use code 1 or 2 for this.',
     'type': 'summary_text'},
    {'text': '**Analyzing statements on negativity**\n\nThe phra

In [9]:
parsed_responses_df

,user_id,is_weakener_depression,is_neutral_depression,is_wkndstrgthnr_depression,is_strength_depression,evidences_depression,is_weakener_symptom,is_neutral_symptom,is_wkndstrgthnr_symptom,is_strength_symptom,evidences_symptom,is_weakener_comorbid,is_neutral_comorbid,is_wkndstrgthnr_comorbid,is_strength_comorbid,evidences_comorbid
0,1,0,1,0,0,"[""Over the past two weeks, I have not been dep...",0,1,1,0,"[""I have been interested in things as much as ...",0,0,0,0,[]
1,2,0,0,1,0,"[""I don't think I have felt depressed."", ""so I...",0,0,1,1,"[""I do find it hard to get pleasure out of thi...",0,0,0,1,"[""Worry, yes, anxiety,""]"
2,3,0,1,0,0,"[""I have been depressed""]",0,0,0,0,[],0,1,0,0,"[""worried about losing my job"", ""doubtful whet..."
3,4,0,1,0,0,"[""i have been feeling depressed""]",0,0,0,0,[],0,0,0,0,[]
4,5,0,0,0,1,"[""yes very much""]",0,0,0,0,[],0,1,1,0,"[""just seem everything is getting on top of me..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
941,972,0,0,0,0,[],0,1,1,1,"[""There has been time when I have not wanted t...",0,0,0,0,[]
942,973,1,0,0,0,"[""I\u2019m not sure if I\u2019ve been depresse...",0,1,0,1,"[""I have definitely been feeling down"", ""Some ...",0,1,0,0,"[""The worry of how the world currently is with..."
943,974,0,1,0,0,"[""I haven\u2019t felt depressed.""]",0,0,0,0,[],1,1,0,0,"[""I have experienced mood swings."", ""This coul..."
944,975,0,1,0,0,"[""I have suffered from depression since my ear...",0,1,1,0,"[""I have no motivation"", ""no motivation or ene...",0,0,0,0,[]


In [30]:
rand_row = random.randint(0, len(parsed_responses_df))
print (responses_df[responses_df.user_id == parsed_responses_df['user_id'].iloc[rand_row]]['user_text'].iloc[0])
parsed_responses_df.iloc[rand_row].to_dict()

I think i am always depressed , my mind is broke . I keep telling people and my therapist that . She changed my meds, i am hoping it works.


{'user_id': 913,
 'is_weakener_depression': 0,
 'is_neutral_depression': 0,
 'is_wkndstrgthnr_depression': 1,
 'is_strength_depression': 0,
 'evidences_depression': '["I think i am always depressed"]',
 'is_weakener_symptom': 0,
 'is_neutral_symptom': 0,
 'is_wkndstrgthnr_symptom': 0,
 'is_strength_symptom': 0,
 'evidences_symptom': '[]',
 'is_weakener_comorbid': 0,
 'is_neutral_comorbid': 0,
 'is_wkndstrgthnr_comorbid': 0,
 'is_strength_comorbid': 0,
 'evidences_comorbid': '[]'}

In [12]:
uncertainity_columns = ['is_weakener_depression', 'is_neutral_depression',	'is_wkndstrgthnr_depression',	'is_strength_depression',
                        'is_weakener_symptom',	'is_neutral_symptom',	'is_wkndstrgthnr_symptom',	'is_strength_symptom',
                        'is_weakener_comorbid',	'is_neutral_comorbid',	'is_wkndstrgthnr_comorbid',	'is_strength_comorbid']


In [13]:
parsed_responses_df[uncertainity_columns].corr(method='pearson')

,is_weakener_depression,is_neutral_depression,is_wkndstrgthnr_depression,is_strength_depression,is_weakener_symptom,is_neutral_symptom,is_wkndstrgthnr_symptom,is_strength_symptom,is_weakener_comorbid,is_neutral_comorbid,is_wkndstrgthnr_comorbid,is_strength_comorbid
is_weakener_depression,1.000000,-0.157969,0.016557,-0.051006,-0.008330,0.007659,-0.008894,0.004877,0.092363,0.020195,0.016260,-0.020325
is_neutral_depression,-0.157969,1.000000,-0.506107,-0.521098,-0.008921,0.057778,-0.014813,-0.062652,0.024760,0.010727,-0.052627,-0.132824
is_wkndstrgthnr_depression,0.016557,-0.506107,1.000000,-0.159249,0.041326,-0.046529,0.072497,-0.029371,-0.012173,0.030026,0.086900,0.049474
is_strength_depression,-0.051006,-0.521098,-0.159249,1.000000,-0.021079,-0.058876,-0.042619,0.095268,-0.045382,-0.089502,0.006029,0.088545
is_weakener_symptom,-0.008330,-0.008921,0.041326,-0.021079,1.000000,-0.044122,-0.009797,-0.011655,-0.009022,0.039312,-0.015825,-0.015379
is_neutral_symptom,0.007659,0.057778,-0.046529,-0.058876,-0.044122,1.000000,0.066164,0.024055,-0.042152,0.055980,-0.019861,-0.102025
is_wkndstrgthnr_symptom,-0.008894,-0.014813,0.072497,-0.042619,-0.009797,0.066164,1.000000,0.120637,0.068293,-0.049483,0.095667,0.032518
is_strength_symptom,0.004877,-0.062652,-0.029371,0.095268,-0.011655,0.024055,0.120637,1.000000,-0.026097,-0.044606,-0.000367,0.063198
is_weakener_comorbid,0.092363,0.024760,-0.012173,-0.045382,-0.009022,-0.042152,0.068293,-0.026097,1.000000,0.088024,0.023682,0.083571
is_neutral_comorbid,0.020195,0.010727,0.030026,-0.089502,0.039312,0.055980,-0.049483,-0.044606,0.088024,1.000000,0.098703,0.057463


In [14]:
self_report_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/self_report_unified.csv'
gpt4_file = '/cronus_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/expt_gpt-4-1106-preview.dep_list_phq9items_score_classify2_editted_unified.csv'
gpt5_file = '/chronos_data/avirinchipur/reasoning_for_psych/expts/parsed_responses/expt_gpt-5.dep_list_phq9items_score_classify2.csv'

In [15]:
gpt4_df = pd.read_csv(gpt4_file)
gpt5_df = pd.read_csv(gpt5_file)
self_report_df = pd.read_csv(self_report_file)

In [16]:
gpt4_df.columns

Index(['user_id', 'user_text', 'score_Anhedonia', 'score_Depressed_Mood',
       'score_Insomnia_or_Hypersomnia', 'score_Fatigue',
       'score_Poor_appetite_or_overeating', 'score_Worthlessness_or_Guilt',
       'score_Difficulty_concentrating',
       'score_Psychomotor_agitation_or_retardation', 'score_Suicidal_ideation',
       'spans_Anhedonia', 'spans_Depressed_Mood',
       'spans_Insomnia_or_Hypersomnia', 'spans_Fatigue',
       'spans_Poor_appetite_or_overeating', 'spans_Worthlessness_or_Guilt',
       'spans_Difficulty_concentrating',
       'spans_Psychomotor_agitation_or_retardation', 'spans_Suicidal_ideation',
       'isInferred_Anhedonia', 'isInferred_Depressed_Mood',
       'isInferred_Insomnia_or_Hypersomnia', 'isInferred_Fatigue',
       'isInferred_Poor_appetite_or_overeating',
       'isInferred_Worthlessness_or_Guilt',
       'isInferred_Difficulty_concentrating',
       'isInferred_Psychomotor_agitation_or_retardation',
       'isInferred_Suicidal_ideation'],
    

In [17]:
score_columns = ['score_Anhedonia', 'score_Depressed_Mood', 'score_Insomnia_or_Hypersomnia', 'score_Fatigue',
       'score_Poor_appetite_or_overeating', 'score_Worthlessness_or_Guilt', 'score_Difficulty_concentrating', 
       'score_Psychomotor_agitation_or_retardation', 'score_Suicidal_ideation']

In [18]:
from scipy.stats import ttest_ind, pearsonr

In [25]:
# First compute the t value between the two distribution for self report scores
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(self_report_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    self_report_series = self_report_df[self_report_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    self_report_series['total_phq'] = self_report_series.iloc[:, 1:].sum(1)
    self_report_series =  self_report_series[['user_id', 'total_phq']]
    merged_df = pd.merge(self_report_series, uncertainity_series, on='user_id')
    control = merged_df[merged_df[col]==0]['total_phq'].values
    treatment = merged_df[merged_df[col]==1]['total_phq'].values
    t_val, p_val = ttest_ind(control, treatment, equal_var=False)
    print (f"{col}: {round(t_val, 3)}({round(p_val, 3)}) | Ns: {len(control)}/{len(treatment)}")

-------------------------------------------
Intersection: 884
is_weakener_depression: 0.225(0.824) | Ns: 854/30
is_neutral_depression: -2.649(0.008) | Ns: 321/563
is_wkndstrgthnr_depression: 3.411(0.001) | Ns: 727/157
is_strength_depression: 0.947(0.345) | Ns: 720/164
-------------------------------------------
Intersection: 494
is_weakener_symptom: 1.873(0.308) | Ns: 492/2
is_neutral_symptom: -0.481(0.633) | Ns: 41/453
is_wkndstrgthnr_symptom: 1.738(0.089) | Ns: 453/41
is_strength_symptom: -3.1(0.003) | Ns: 437/57
-------------------------------------------
Intersection: 607
is_weakener_comorbid: -0.999(0.324) | Ns: 572/35
is_neutral_comorbid: -1.024(0.309) | Ns: 60/547
is_wkndstrgthnr_comorbid: 1.694(0.092) | Ns: 507/100
is_strength_comorbid: -0.636(0.526) | Ns: 512/95


In [26]:
# First compute the t value between the two distribution
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt4_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt4_series = gpt4_df[gpt4_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt4_series['total_phq'] = gpt4_series.iloc[:, 1:].sum(1)
    gpt4_series =  gpt4_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt4_series, uncertainity_series, on='user_id')
    control = merged_df[merged_df[col]==0]['total_phq'].values
    treatment = merged_df[merged_df[col]==1]['total_phq'].values
    t_val, p_val = ttest_ind(control, treatment, equal_var=False)
    print (f"{col}: {round(t_val, 3)}({round(p_val, 3)}) | Ns: {len(control)}/{len(treatment)}")

-------------------------------------------
Intersection: 884
is_weakener_depression: 0.515(0.61) | Ns: 854/30
is_neutral_depression: -2.685(0.007) | Ns: 321/563
is_wkndstrgthnr_depression: 4.369(0.0) | Ns: 727/157
is_strength_depression: 0.296(0.768) | Ns: 720/164
-------------------------------------------
Intersection: 494
is_weakener_symptom: 1.878(0.305) | Ns: 492/2
is_neutral_symptom: -0.375(0.709) | Ns: 41/453
is_wkndstrgthnr_symptom: 1.938(0.059) | Ns: 453/41
is_strength_symptom: -3.844(0.0) | Ns: 437/57
-------------------------------------------
Intersection: 607
is_weakener_comorbid: 0.364(0.718) | Ns: 572/35
is_neutral_comorbid: -1.903(0.061) | Ns: 60/547
is_wkndstrgthnr_comorbid: 2.571(0.011) | Ns: 507/100
is_strength_comorbid: 0.243(0.808) | Ns: 512/95


In [27]:
# First compute the t value between the two distribution
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt5_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt5_series = gpt5_df[gpt5_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt5_series['total_phq'] = gpt5_series.iloc[:, 1:].sum(1)
    gpt5_series =  gpt5_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt5_series, uncertainity_series, on='user_id')
    control = merged_df[merged_df[col]==0]['total_phq'].values
    treatment = merged_df[merged_df[col]==1]['total_phq'].values
    t_val, p_val = ttest_ind(control, treatment, equal_var=False)
    print (f"{col}: {round(t_val, 3)}({round(p_val, 3)}) | Ns: {len(control)}/{len(treatment)}")

-------------------------------------------
Intersection: 884
is_weakener_depression: 0.849(0.402) | Ns: 854/30
is_neutral_depression: -2.654(0.008) | Ns: 321/563
is_wkndstrgthnr_depression: 4.521(0.0) | Ns: 727/157
is_strength_depression: 0.039(0.969) | Ns: 720/164
-------------------------------------------
Intersection: 494
is_weakener_symptom: 3.406(0.167) | Ns: 492/2
is_neutral_symptom: -1.154(0.254) | Ns: 41/453
is_wkndstrgthnr_symptom: 1.62(0.112) | Ns: 453/41
is_strength_symptom: -2.991(0.004) | Ns: 437/57
-------------------------------------------
Intersection: 607
is_weakener_comorbid: 0.241(0.811) | Ns: 572/35
is_neutral_comorbid: -2.546(0.013) | Ns: 60/547
is_wkndstrgthnr_comorbid: 3.428(0.001) | Ns: 507/100
is_strength_comorbid: 1.759(0.081) | Ns: 512/95


In [33]:
def cohen_d_unequal_samples(data1, data2):
    """
    Calculates Cohen's d for two independent samples with potentially unequal sizes.

    Args:
        data1 (array-like): The first sample data.
        data2 (array-like): The second sample data.

    Returns:
        float: The calculated Cohen's d value.
    """
    n1, n2 = len(data1), len(data2)
    mean1, mean2 = np.mean(data1), np.mean(data2)
    var1, var2 = np.var(data1, ddof=1), np.var(data2, ddof=1) # ddof=1 for sample variance

    # Calculate the pooled standard deviation
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))

    # Calculate Cohen's d
    cohens_d = (mean1 - mean2) / pooled_std
    return cohens_d

In [34]:
# First compute the t value between the two distribution
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt4_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt4_series = gpt4_df[gpt4_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt4_series['total_phq'] = gpt4_series.iloc[:, 1:].sum(1)
    gpt4_series =  gpt4_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt4_series, uncertainity_series, on='user_id')
    control = merged_df[merged_df[col]==0]['total_phq'].values
    treatment = merged_df[merged_df[col]==1]['total_phq'].values
    _, p_val = ttest_ind(control, treatment, equal_var=False)
    d_val = cohen_d_unequal_samples(control, treatment)
    print (f"{col}: {round(d_val, 3)}({round(p_val, 3)}) | Ns: {len(control)}/{len(treatment)}")

-------------------------------------------
Intersection: 884
is_weakener_depression: 0.079(0.61) | Ns: 854/30
is_neutral_depression: -0.187(0.007) | Ns: 321/563
is_wkndstrgthnr_depression: 0.327(0.0) | Ns: 727/157
is_strength_depression: 0.028(0.768) | Ns: 720/164
-------------------------------------------
Intersection: 494
is_weakener_symptom: 0.631(0.305) | Ns: 492/2
is_neutral_symptom: -0.063(0.709) | Ns: 41/453
is_wkndstrgthnr_symptom: 0.335(0.059) | Ns: 453/41
is_strength_symptom: -0.503(0.0) | Ns: 437/57
-------------------------------------------
Intersection: 607
is_weakener_comorbid: 0.064(0.718) | Ns: 572/35
is_neutral_comorbid: -0.283(0.061) | Ns: 60/547
is_wkndstrgthnr_comorbid: 0.273(0.011) | Ns: 507/100
is_strength_comorbid: 0.028(0.808) | Ns: 512/95


In [35]:
# First compute the t value between the two distribution
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt5_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt5_series = gpt5_df[gpt5_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt5_series['total_phq'] = gpt5_series.iloc[:, 1:].sum(1)
    gpt5_series =  gpt5_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt5_series, uncertainity_series, on='user_id')
    control = merged_df[merged_df[col]==0]['total_phq'].values
    treatment = merged_df[merged_df[col]==1]['total_phq'].values
    _, p_val = ttest_ind(control, treatment, equal_var=False)
    d_val = cohen_d_unequal_samples(control, treatment)
    print (f"{col}: {round(d_val, 3)}({round(p_val, 3)}) | Ns: {len(control)}/{len(treatment)}")

-------------------------------------------
Intersection: 884
is_weakener_depression: 0.122(0.402) | Ns: 854/30
is_neutral_depression: -0.181(0.008) | Ns: 321/563
is_wkndstrgthnr_depression: 0.335(0.0) | Ns: 727/157
is_strength_depression: 0.004(0.969) | Ns: 720/164
-------------------------------------------
Intersection: 494
is_weakener_symptom: 0.794(0.167) | Ns: 492/2
is_neutral_symptom: -0.181(0.254) | Ns: 41/453
is_wkndstrgthnr_symptom: 0.267(0.112) | Ns: 453/41
is_strength_symptom: -0.425(0.004) | Ns: 437/57
-------------------------------------------
Intersection: 607
is_weakener_comorbid: 0.043(0.811) | Ns: 572/35
is_neutral_comorbid: -0.322(0.013) | Ns: 60/547
is_wkndstrgthnr_comorbid: 0.309(0.001) | Ns: 507/100
is_strength_comorbid: 0.186(0.081) | Ns: 512/95


In [28]:
# Compute the convergent validity under each setting
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt4_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt4_series = gpt4_df[gpt4_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt4_series['total_phq'] = gpt4_series.iloc[:, 1:].sum(1)
    gpt4_series =  gpt4_series[['user_id', 'total_phq']]
    self_report_series = self_report_df[['user_id']+score_columns]
    self_report_series['total_phq'] = self_report_df[score_columns].sum(1)
    self_report_series = self_report_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt4_series, uncertainity_series, on='user_id').merge(self_report_series, on='user_id', suffixes=('_gpt', '_sr'))
    control = merged_df[merged_df[col]==0][['total_phq_gpt', 'total_phq_sr']]
    control_r, control_p = pearsonr(control['total_phq_gpt'], control['total_phq_sr'])
    treatment = merged_df[merged_df[col]==1][['total_phq_gpt', 'total_phq_sr']]
    treatment_r, treatment_p = pearsonr(treatment['total_phq_gpt'], treatment['total_phq_sr'])
    print (f"{col} | Ns: {len(control)}/{len(treatment)} | {round(control_r, 3)}({round(control_p, 2)}) - {round(treatment_r, 3)}({round(treatment_p, 2)})")

-------------------------------------------
Intersection: 884
is_weakener_depression | Ns: 854/30 | 0.733(0.0) - 0.513(0.0)
is_neutral_depression | Ns: 321/563 | 0.739(0.0) - 0.719(0.0)
is_wkndstrgthnr_depression | Ns: 727/157 | 0.738(0.0) - 0.638(0.0)
is_strength_depression | Ns: 720/164 | 0.701(0.0) - 0.815(0.0)
-------------------------------------------
Intersection: 494
is_weakener_symptom | Ns: 492/2 | 0.735(0.0) - 1.0(1.0)
is_neutral_symptom | Ns: 41/453 | 0.755(0.0) - 0.734(0.0)
is_wkndstrgthnr_symptom | Ns: 453/41 | 0.733(0.0) - 0.74(0.0)
is_strength_symptom | Ns: 437/57 | 0.731(0.0) - 0.726(0.0)
-------------------------------------------
Intersection: 607
is_weakener_comorbid | Ns: 572/35 | 0.681(0.0) - 0.747(0.0)
is_neutral_comorbid | Ns: 60/547 | 0.67(0.0) - 0.685(0.0)
is_wkndstrgthnr_comorbid | Ns: 507/100 | 0.676(0.0) - 0.714(0.0)
is_strength_comorbid | Ns: 512/95 | 0.689(0.0) - 0.66(0.0)


In [29]:

# Compute the convergent validity under each setting
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt5_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt5_series = gpt5_df[gpt5_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt5_series['total_phq'] = gpt5_series.iloc[:, 1:].sum(1)
    gpt5_series =  gpt5_series[['user_id', 'total_phq']]
    self_report_series = self_report_df[['user_id']+score_columns]
    self_report_series['total_phq'] = self_report_df[score_columns].sum(1)
    self_report_series = self_report_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt5_series, uncertainity_series, on='user_id').merge(self_report_series, on='user_id', suffixes=('_gpt', '_sr'))
    control = merged_df[merged_df[col]==0][['total_phq_gpt', 'total_phq_sr']]
    control_r, control_p = pearsonr(control['total_phq_gpt'], control['total_phq_sr'])
    treatment = merged_df[merged_df[col]==1][['total_phq_gpt', 'total_phq_sr']]
    treatment_r, treatment_p = pearsonr(treatment['total_phq_gpt'], treatment['total_phq_sr'])
    print (f"{col} | Ns: {len(control)}/{len(treatment)} | {round(control_r, 3)}({round(control_p, 2)}) - {round(treatment_r, 3)}({round(treatment_p, 2)})")

-------------------------------------------
Intersection: 884
is_weakener_depression | Ns: 854/30 | 0.703(0.0) - 0.616(0.0)
is_neutral_depression | Ns: 321/563 | 0.712(0.0) - 0.693(0.0)
is_wkndstrgthnr_depression | Ns: 727/157 | 0.707(0.0) - 0.634(0.0)
is_strength_depression | Ns: 720/164 | 0.687(0.0) - 0.752(0.0)
-------------------------------------------
Intersection: 494
is_weakener_symptom | Ns: 492/2 | 0.685(0.0) - 1.0(1.0)
is_neutral_symptom | Ns: 41/453 | 0.692(0.0) - 0.686(0.0)
is_wkndstrgthnr_symptom | Ns: 453/41 | 0.68(0.0) - 0.735(0.0)
is_strength_symptom | Ns: 437/57 | 0.686(0.0) - 0.636(0.0)
-------------------------------------------
Intersection: 607
is_weakener_comorbid | Ns: 572/35 | 0.661(0.0) - 0.707(0.0)
is_neutral_comorbid | Ns: 60/547 | 0.728(0.0) - 0.656(0.0)
is_wkndstrgthnr_comorbid | Ns: 507/100 | 0.662(0.0) - 0.667(0.0)
is_strength_comorbid | Ns: 512/95 | 0.662(0.0) - 0.69(0.0)


In [30]:
# Compute the convergent validity under each setting
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt4_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt4_series = gpt4_df[gpt4_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt4_series['total_phq'] = gpt4_series.iloc[:, 1:].sum(1)
    gpt4_series =  gpt4_series[['user_id', 'total_phq']]
    self_report_series = self_report_df[['user_id']+score_columns]
    self_report_series['total_phq'] = self_report_df[score_columns].sum(1)
    self_report_series = self_report_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt4_series, uncertainity_series, on='user_id').merge(self_report_series, on='user_id', suffixes=('_gpt', '_sr'))
    control = merged_df[merged_df[col]==0][['total_phq_gpt', 'total_phq_sr']]
    control_err = (control['total_phq_sr'] - control['total_phq_gpt']) 
    treatment = merged_df[merged_df[col]==1][['total_phq_gpt', 'total_phq_sr']]
    treatment_err = (treatment['total_phq_sr'] - treatment['total_phq_gpt'])
    t_val, p_val = ttest_ind(control_err, treatment_err, equal_var=False)
    print (f"{col} | Ns: {len(control)}/{len(treatment)} | {round(t_val, 3)}({round(p_val, 2)})")

-------------------------------------------
Intersection: 884
is_weakener_depression | Ns: 854/30 | -0.146(0.89)
is_neutral_depression | Ns: 321/563 | -1.403(0.16)
is_wkndstrgthnr_depression | Ns: 727/157 | 1.106(0.27)
is_strength_depression | Ns: 720/164 | 1.218(0.22)
-------------------------------------------
Intersection: 494
is_weakener_symptom | Ns: 492/2 | 1.853(0.31)
is_neutral_symptom | Ns: 41/453 | -0.368(0.71)
is_wkndstrgthnr_symptom | Ns: 453/41 | 0.626(0.53)
is_strength_symptom | Ns: 437/57 | -1.159(0.25)
-------------------------------------------
Intersection: 607
is_weakener_comorbid | Ns: 572/35 | -1.746(0.09)
is_neutral_comorbid | Ns: 60/547 | 0.279(0.78)
is_wkndstrgthnr_comorbid | Ns: 507/100 | 0.199(0.84)
is_strength_comorbid | Ns: 512/95 | -1.055(0.29)


In [31]:
# Compute the convergent validity under each setting
for idx, col in enumerate(uncertainity_columns):
    # Filter rows out that have all 0s for that subject
    subject_cols = uncertainity_columns[(idx//4)*4:(idx//4)*4+4]
    nonnull_users = parsed_responses_df[['user_id']+subject_cols]
    nonnull_users = nonnull_users[nonnull_users[subject_cols].sum(1)!=0].user_id.tolist()
    temp_user_ids = set(nonnull_users).intersection(gpt5_df.user_id.tolist())
    if idx%4==0: 
        print ('-------------------------------------------')
        print (f'Intersection: {len(nonnull_users)}')
    uncertainity_series =  parsed_responses_df[parsed_responses_df.user_id.isin(temp_user_ids)][['user_id', col]]
    gpt5_series = gpt5_df[gpt5_df.user_id.isin(temp_user_ids)][['user_id']+ score_columns]
    gpt5_series['total_phq'] = gpt5_series.iloc[:, 1:].sum(1)
    gpt5_series =  gpt5_series[['user_id', 'total_phq']]
    self_report_series = self_report_df[['user_id']+score_columns]
    self_report_series['total_phq'] = self_report_df[score_columns].sum(1)
    self_report_series = self_report_series[['user_id', 'total_phq']]
    merged_df = pd.merge(gpt5_series, uncertainity_series, on='user_id').merge(self_report_series, on='user_id', suffixes=('_gpt', '_sr'))
    control = merged_df[merged_df[col]==0][['total_phq_gpt', 'total_phq_sr']]
    control_err = (control['total_phq_sr'] - control['total_phq_gpt']) 
    treatment = merged_df[merged_df[col]==1][['total_phq_gpt', 'total_phq_sr']]
    treatment_err = (treatment['total_phq_sr'] - treatment['total_phq_gpt'])
    t_val, p_val = ttest_ind(control_err, treatment_err, equal_var=False)
    print (f"{col} | Ns: {len(control)}/{len(treatment)} | {round(t_val, 3)}({round(p_val, 2)})")

-------------------------------------------
Intersection: 884
is_weakener_depression | Ns: 854/30 | -0.333(0.74)
is_neutral_depression | Ns: 321/563 | -1.631(0.1)
is_wkndstrgthnr_depression | Ns: 727/157 | 1.355(0.18)
is_strength_depression | Ns: 720/164 | 1.322(0.19)
-------------------------------------------
Intersection: 494
is_weakener_symptom | Ns: 492/2 | 1.083(0.47)
is_neutral_symptom | Ns: 41/453 | 0.239(0.81)
is_wkndstrgthnr_symptom | Ns: 453/41 | 1.045(0.3)
is_strength_symptom | Ns: 437/57 | -1.588(0.12)
-------------------------------------------
Intersection: 607
is_weakener_comorbid | Ns: 572/35 | -1.554(0.13)
is_neutral_comorbid | Ns: 60/547 | 0.438(0.66)
is_wkndstrgthnr_comorbid | Ns: 507/100 | 0.019(0.99)
is_strength_comorbid | Ns: 512/95 | -2.186(0.03)
